# CNN-GRU sleep-stage classifier

This notebook adapts the attached GRU paper's architecture: two stacked bidirectional GRU layers followed by a softmax classifier. The raw 1 Hz Polar heart-rate sequence is passed directly to the recurrent model. Each one-second slice of the two 20 Hz FDC channels is first encoded by a small 1-D CNN into one 1 Hz feature vector. The resulting 1 Hz HR plus FDC representation is classified for a 30-second center epoch using five preceding and four following epochs as context.

The Apple Watch stages are used only as labels. One complete sleep is held out for validation. Inputs are standardized using training sleeps only.

In [7]:
from __future__ import annotations

import copy
import math
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

DATASET_DIR = Path.cwd()
if DATASET_DIR.name != "datasets_for_training":
    raise RuntimeError("Open this notebook with datasets_for_training as the working directory.")
VALIDATION_DATE = "2026-10-20"
SAMPLE_RATE = 20
EPOCH_SECONDS = 30
CONTEXT_BEFORE = 5
CONTEXT_AFTER = 4
CONTEXT_EPOCHS = CONTEXT_BEFORE + 1 + CONTEXT_AFTER
SEQUENCE_SECONDS = CONTEXT_EPOCHS * EPOCH_SECONDS
BATCH_SIZE = 100
GRU_HIDDEN_SIZE = 128
GRU_LAYERS = 2
DROPOUT = 0.1
LEARNING_RATE = 1e-4
MAX_EPOCHS = 300
LR_PATIENCE = 10
EARLY_STOPPING_PATIENCE = 30
STAGES = ["Awake", "REM", "Core", "Deep"]
STAGE_TO_INDEX = {stage: index for index, stage in enumerate(STAGES)}
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required, but torch.cuda.is_available() is False.")
DEVICE = torch.device("cuda")
print("Device:", torch.cuda.get_device_name(DEVICE))
print("PyTorch:", torch.__version__)
print("Validation sleep:", VALIDATION_DATE)

Device: NVIDIA GeForce RTX 3050 6GB Laptop GPU
PyTorch: 2.11.0+cu128
Validation sleep: 2026-10-20


## Loading and aligning the training files

In [8]:
def _date_from_name(path: Path) -> str:
    match = re.fullmatch(r"(\d{4}-\d{2}-\d{2})", path.stem)
    if match is None:
        raise ValueError(f"Unexpected Apple Watch filename: {path.name}")
    return match.group(1)


def _find_files(recording_date: str) -> tuple[Path, Path, Path]:
    apple = DATASET_DIR / f"{recording_date}.csv"
    fdc = DATASET_DIR / f"fdc1004_data_{recording_date.replace('-', '_')}.csv"
    polar = sorted(DATASET_DIR.glob(f"polar_{recording_date}_*.CSV"))
    if not apple.is_file() or not fdc.is_file() or not polar:
        raise FileNotFoundError(f"Missing one or more files for {recording_date}")
    return apple, polar[0], fdc


def _load_sleep(recording_date: str) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    apple_path, polar_path, fdc_path = _find_files(recording_date)
    apple = pd.read_csv(apple_path)
    for column in ("Start time", "End time"):
        apple[column] = pd.to_datetime(apple[column]).map(
            lambda value: value.tz_localize(None) if value.tzinfo else value
        )
    apple_start = apple["Start time"].min()
    apple["start_s"] = (apple["Start time"] - apple_start).dt.total_seconds()
    apple["end_s"] = (apple["End time"] - apple_start).dt.total_seconds()

    polar = pd.read_csv(polar_path)
    polar["time_s"] = pd.to_timedelta(polar["Time"]).dt.total_seconds()
    polar["time_s"] -= polar["time_s"].min()
    polar["HR (bpm)"] = pd.to_numeric(polar["HR (bpm)"], errors="coerce")
    polar = polar.dropna(subset=["time_s", "HR (bpm)"]).drop_duplicates("time_s")

    fdc = pd.read_csv(fdc_path, usecols=["timestamp_ms", "CIN2_pF", "CIN4_pF"])
    timestamp_ms = pd.to_numeric(fdc["timestamp_ms"], errors="coerce")
    fdc["time_s"] = (timestamp_ms - timestamp_ms.iloc[0]) / 1000
    for column in ("CIN2_pF", "CIN4_pF"):
        fdc[column] = pd.to_numeric(fdc[column], errors="coerce")
    fdc = fdc.dropna(subset=["time_s", "CIN2_pF", "CIN4_pF"])

    duration = min(float(apple["end_s"].max()), float(polar["time_s"].max()), float(fdc["time_s"].max()))
    seconds = np.arange(0, math.floor(duration), dtype=np.float32)
    hr = np.interp(seconds, polar["time_s"], polar["HR (bpm)"]).astype(np.float32)
    fdc_samples = (seconds[:, None] + np.arange(SAMPLE_RATE, dtype=np.float32)[None, :] / SAMPLE_RATE).reshape(-1)
    fdc_values = np.column_stack([
        np.interp(fdc_samples, fdc["time_s"], fdc["CIN2_pF"]),
        np.interp(fdc_samples, fdc["time_s"], fdc["CIN4_pF"]),
    ]).reshape(len(seconds), SAMPLE_RATE, 2).astype(np.float32)
    labels = np.full(len(seconds), -1, dtype=np.int64)
    for row in apple.itertuples(name=None, index=False):
        mask = (seconds >= row[3]) & (seconds < row[4])
        labels[mask] = STAGE_TO_INDEX[row[0]]
    valid = labels >= 0
    return hr[valid], fdc_values[valid], labels[valid]


recording_dates = sorted(_date_from_name(path) for path in DATASET_DIR.glob("20??-??-??.csv"))
if len(recording_dates) != 4 or VALIDATION_DATE not in recording_dates:
    raise ValueError(f"Expected four sleeps including {VALIDATION_DATE}; found {recording_dates}")
sleep_data = {recording_date: _load_sleep(recording_date) for recording_date in recording_dates}
for recording_date, data in sleep_data.items():
    print(recording_date, len(data[0]), "one-Hz steps", data[1].shape)

2026-10-16 21444 one-Hz steps (21444, 20, 2)
2026-10-17 21093 one-Hz steps (21093, 20, 2)
2026-10-19 19867 one-Hz steps (19867, 20, 2)
2026-10-20 8834 one-Hz steps (8834, 20, 2)


C:\Users\Roli\AppData\Local\Temp\ipykernel_43556\2504001156.py:34: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  fdc = pd.read_csv(fdc_path, usecols=["timestamp_ms", "CIN2_pF", "CIN4_pF"])


## Context sequences and labels

FDC is retained at 20 Hz inside every one-second step. The CNN receives `(CIN2, CIN4)` with 20 samples and emits one vector. HR is kept as the original 1 Hz value. The target is the majority Apple Watch class in the center 30-second epoch.

In [9]:
def _make_examples(data: tuple[np.ndarray, np.ndarray, np.ndarray]) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    usable = len(data[0]) // EPOCH_SECONDS * EPOCH_SECONDS
    hr, fdc, labels = data
    hr, fdc, labels = hr[:usable], fdc[:usable], labels[:usable]
    epoch_labels = []
    for start in range(0, usable, EPOCH_SECONDS):
        epoch_labels.append(int(np.bincount(labels[start:start + EPOCH_SECONDS], minlength=len(STAGES)).argmax()))
    hr_examples, fdc_examples, targets = [], [], []
    for center in range(CONTEXT_BEFORE, len(epoch_labels) - CONTEXT_AFTER):
        start = (center - CONTEXT_BEFORE) * EPOCH_SECONDS
        stop = (center + CONTEXT_AFTER + 1) * EPOCH_SECONDS
        hr_examples.append(hr[start:stop])
        fdc_examples.append(fdc[start:stop].transpose(0, 2, 1))
        targets.append(epoch_labels[center])
    return np.asarray(hr_examples), np.asarray(fdc_examples), np.asarray(targets, dtype=np.int64)


def _standardize(train: np.ndarray, other: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    mean = train.reshape(-1, train.shape[-1]).mean(axis=0)
    std = train.reshape(-1, train.shape[-1]).std(axis=0)
    std[std < 1e-6] = 1.0
    return (train - mean) / std, (other - mean) / std


train_parts = [_make_examples(sleep_data[d]) for d in recording_dates if d != VALIDATION_DATE]
validation_hr, validation_fdc, validation_y = _make_examples(sleep_data[VALIDATION_DATE])
train_hr = np.concatenate([part[0] for part in train_parts])
train_fdc = np.concatenate([part[1] for part in train_parts])
train_y = np.concatenate([part[2] for part in train_parts])
train_hr, validation_hr = _standardize(train_hr[..., None], validation_hr[..., None])
train_hr, validation_hr = train_hr[..., 0], validation_hr[..., 0]
fdc_mean = train_fdc.reshape(-1, 2).mean(axis=0)
fdc_std = train_fdc.reshape(-1, 2).std(axis=0)
fdc_std[fdc_std < 1e-6] = 1.0
fdc_mean = fdc_mean.reshape(1, 1, 2, 1)
fdc_std = fdc_std.reshape(1, 1, 2, 1)
train_fdc = (train_fdc - fdc_mean) / fdc_std
validation_fdc = (validation_fdc - fdc_mean) / fdc_std
print("Train HR/FDC:", train_hr.shape, train_fdc.shape)
print("Validation HR/FDC:", validation_hr.shape, validation_fdc.shape)

Train HR/FDC: (2052, 300) (2052, 300, 2, 20)
Validation HR/FDC: (285, 300) (285, 300, 2, 20)


In [10]:
class SleepSequenceDataset(Dataset):
    def __init__(self, hr: np.ndarray, fdc: np.ndarray, labels: np.ndarray) -> None:
        self.hr = torch.from_numpy(hr.astype(np.float32))
        self.fdc = torch.from_numpy(fdc.astype(np.float32))
        self.labels = torch.from_numpy(labels)

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        return self.hr[index], self.fdc[index], self.labels[index]


class FDCEncoder(nn.Module):
    def __init__(self, output_size: int = 32) -> None:
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv1d(2, 16, kernel_size=5, padding=2),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(16, output_size, kernel_size=3, padding=1),
            nn.BatchNorm1d(output_size),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )

    def forward(self, fdc: torch.Tensor) -> torch.Tensor:
        batch, steps, channels, samples = fdc.shape
        encoded = self.network(fdc.reshape(batch * steps, channels, samples))
        return encoded.flatten(1).reshape(batch, steps, -1)


class CNNGRUClassifier(nn.Module):
    def __init__(self, classes: int = 4) -> None:
        super().__init__()
        self.fdc_encoder = FDCEncoder()
        self.gru = nn.GRU(
            input_size=33,
            hidden_size=GRU_HIDDEN_SIZE,
            num_layers=GRU_LAYERS,
            batch_first=True,
            bidirectional=True,
            dropout=DROPOUT,
        )
        self.classifier = nn.Linear(GRU_HIDDEN_SIZE * 2, classes)
        self.apply(self._initialize)

    @staticmethod
    def _initialize(module: nn.Module) -> None:
        if isinstance(module, (nn.Conv1d, nn.Linear)):
            nn.init.kaiming_normal_(module.weight, mode="fan_in", nonlinearity="relu")
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        if isinstance(module, nn.GRU):
            for name, parameter in module.named_parameters():
                if "weight" in name:
                    nn.init.xavier_uniform_(parameter)
                elif "bias" in name:
                    nn.init.zeros_(parameter)

    def forward(self, hr: torch.Tensor, fdc: torch.Tensor) -> torch.Tensor:
        fdc_features = self.fdc_encoder(fdc)
        sequence, _ = self.gru(torch.cat((hr.unsqueeze(-1), fdc_features), dim=-1))
        center_start = CONTEXT_BEFORE * EPOCH_SECONDS
        center_end = center_start + EPOCH_SECONDS
        return self.classifier(sequence[:, center_start:center_end].mean(dim=1))


model = CNNGRUClassifier().to(DEVICE)
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=LR_PATIENCE)
train_loader = DataLoader(SleepSequenceDataset(train_hr, train_fdc, train_y), batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
validation_loader = DataLoader(SleepSequenceDataset(validation_hr, validation_fdc, validation_y), batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)
print(model)

CNNGRUClassifier(
  (fdc_encoder): FDCEncoder(
    (network): Sequential(
      (0): Conv1d(2, 16, kernel_size=(5,), stride=(1,), padding=(2,))
      (1): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (4): Conv1d(16, 32, kernel_size=(3,), stride=(1,), padding=(1,))
      (5): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (6): ReLU()
      (7): AdaptiveAvgPool1d(output_size=1)
    )
  )
  (gru): GRU(33, 128, num_layers=2, batch_first=True, dropout=0.1, bidirectional=True)
  (classifier): Linear(in_features=256, out_features=4, bias=True)
)


In [11]:
def _evaluate(loader: DataLoader) -> tuple[float, np.ndarray]:
    model.eval()
    total_loss, predictions, targets = 0.0, [], []
    with torch.no_grad():
        for hr, fdc, labels in loader:
            hr = hr.to(DEVICE, non_blocking=True)
            fdc = fdc.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            logits = model(hr, fdc)
            total_loss += loss_function(logits, labels).item() * labels.size(0)
            predictions.append(logits.argmax(1).cpu().numpy())
            targets.append(labels.cpu().numpy())
    return total_loss / len(loader.dataset), np.column_stack((np.concatenate(targets), np.concatenate(predictions)))


history, best_state, best_loss, stale = [], None, float("inf"), 0
for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    for hr, fdc, labels in train_loader:
        hr = hr.to(DEVICE, non_blocking=True)
        fdc = fdc.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        loss_function(model(hr, fdc), labels).backward()
        optimizer.step()
    validation_loss, _ = _evaluate(validation_loader)
    scheduler.step(validation_loss)
    history.append((epoch, validation_loss, optimizer.param_groups[0]["lr"]))
    if validation_loss < best_loss - 1e-6:
        best_loss, best_state, stale = validation_loss, copy.deepcopy(model.state_dict()), 0
    else:
        stale += 1
    if epoch == 1 or epoch % 10 == 0:
        print(f"epoch={epoch:03d} validation_loss={validation_loss:.5f} lr={optimizer.param_groups[0]['lr']:.2e}")
    if stale >= EARLY_STOPPING_PATIENCE:
        print("Early stopping at epoch", epoch)
        break
model.load_state_dict(best_state)
print("Best validation loss:", best_loss)

epoch=001 validation_loss=1.02610 lr=1.00e-04
epoch=010 validation_loss=1.05715 lr=1.00e-04
epoch=020 validation_loss=1.10849 lr=5.00e-05
epoch=030 validation_loss=1.30255 lr=2.50e-05
Early stopping at epoch 32
Best validation loss: 1.0047679093846105


In [12]:
def classification_report(targets: np.ndarray, predictions: np.ndarray) -> pd.DataFrame:
    matrix = np.zeros((len(STAGES), len(STAGES)), dtype=np.int64)
    for target, prediction in zip(targets, predictions):
        matrix[target, prediction] += 1
    rows = []
    for index, stage in enumerate(STAGES):
        tp = matrix[index, index]
        precision = tp / matrix[:, index].sum() if matrix[:, index].sum() else 0.0
        recall = tp / matrix[index].sum() if matrix[index].sum() else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        rows.append({"stage": stage, "precision": precision, "recall": recall, "f1": f1, "support": matrix[index].sum()})
    total = matrix.sum()
    accuracy = np.trace(matrix) / total if total else 0.0
    expected = (matrix.sum(axis=0) * matrix.sum(axis=1)).sum() / total**2 if total else 0.0
    kappa = (accuracy - expected) / (1 - expected) if expected < 1 else 0.0
    print("Accuracy:", accuracy, "Cohen kappa:", kappa)
    print("Confusion matrix (rows=true, columns=predicted):")
    print(pd.DataFrame(matrix, index=STAGES, columns=STAGES))
    return pd.DataFrame(rows).set_index("stage")


validation_loss, result = _evaluate(validation_loader)
report = classification_report(result[:, 0], result[:, 1])
print(report)
print("TST ground truth (hours):", (result[:, 0] != STAGE_TO_INDEX["Awake"]).sum() * EPOCH_SECONDS / 3600)
print("TST CNN-GRU (hours):", (result[:, 1] != STAGE_TO_INDEX["Awake"]).sum() * EPOCH_SECONDS / 3600)

Accuracy: 0.5263157894736842 Cohen kappa: -0.045232273838630765
Confusion matrix (rows=true, columns=predicted):
       Awake  REM  Core  Deep
Awake      0    0     0     0
REM        0    2    21     4
Core       0    5   140    25
Deep       0    0    80     8
       precision    recall        f1  support
stage                                        
Awake   0.000000  0.000000  0.000000        0
REM     0.285714  0.074074  0.117647       27
Core    0.580913  0.823529  0.681265      170
Deep    0.216216  0.090909  0.128000       88
TST ground truth (hours): 2.375
TST CNN-GRU (hours): 2.375
